# Quranic SQL Basics with DuckDB CLI

Simple SQL queries for exploring the JSON datasets in `quran-data`.

---

## Datasets

- `surah.json`: Surah details and revelation place.
- `ayah.json`: Verse text and word counts.
- `juz.json`: Juz divisions and verse counts.
- `sajda.json`: Prostration verses.
- `matching-ayah.json`: Nested verse similarity matches.

## Run the SQL with DuckDB CLI

Each SQL cell starts DuckDB CLI with `%%script duckdb`. The SQL body is sent directly to the CLI, so the cells can be run from this notebook without creating views.

### Question 1: Surahs and Verses by Revelation Place

Count total surahs and total verses for Makkah vs Madinah, including each place's percentage of the total.

**SQL Concepts:** `WITH`, `GROUP BY`, `COUNT`, `SUM`, window functions.

In [21]:
%%script duckdb
-- Count surahs and verses, then calculate each place's percentage of the total.
-- A CTE is a temporary named result used to organize a query.
WITH surah_stats AS (
    SELECT
        revelation_place,
        COUNT(*) AS total_surahs,
        SUM(verses_count) AS total_verses
    FROM 'quran-data/surah.json'
    GROUP BY revelation_place
)
SELECT
    revelation_place,
    total_surahs,
    -- Overall percentage of surahs and verses by revelation place.
    -- SUM(...) OVER () totals all groups while keeping each group as a row.
    ROUND(100.0 * total_surahs / SUM(total_surahs) OVER (), 2) AS surah_percentage,
    total_verses,
    ROUND(100.0 * total_verses / SUM(total_verses) OVER (), 2) AS verse_percentage
FROM surah_stats
ORDER BY total_surahs DESC;

┌──────────────────┬──────────────┬──────────────────┬──────────────┬──────────────────┐
│ revelation_place │ total_surahs │ surah_percentage │ total_verses │ verse_percentage │
│     varchar      │    int64     │      double      │    int128    │      double      │
├──────────────────┼──────────────┼──────────────────┼──────────────┼──────────────────┤
│ Makkah           │           86 │            75.44 │         4613 │            73.97 │
│ Madinah          │           28 │            24.56 │         1623 │            26.03 │
└──────────────────┴──────────────┴──────────────────┴──────────────┴──────────────────┘


### Question 2: Surahs with Prostration Verses (Sajdah)

List Sajdah verses with their Surah names and Arabic text.

**Summary:** Start with Sajdah records, join each verse to `ayah.json` for its text, then join to `surah.json` for the Surah names.

**SQL Concepts:** `JOIN` combines related rows; `ORDER BY` controls the display order.

In [22]:
%%script duckdb
-- Start with Sajdah rows, then add verse text and Surah names.
SELECT
    s.sajdah_number,
    s.verse_key,
    s.sajdah_type,
    su.name_arabic AS surah_name,
    su.name_english AS surah_name_english,
    a.text AS verse_text
FROM 'quran-data/sajda.json' AS s
-- JOIN ayahs by verse_key to get the text and surah_number.
JOIN 'quran-data/ayah.json' AS a
    ON s.verse_key = a.verse_key
-- JOIN surahs by id to get the Surah names.
JOIN 'quran-data/surah.json' AS su
    ON a.surah_number = su.id
-- Sort the output by the Sajdah number.
ORDER BY s.sajdah_number;

┌───────────────┬───────────┬─────────────┬────────────┬────────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│ sajdah_number │ verse_key │ sajdah_type │ surah_name │ surah_name_english │                                                                                      verse_text                                                                                       │
│     int64     │  varchar  │   varchar   │  varchar   │      varchar       │                                                                                        varchar                                                                                        │
├───────────────┼───────────┼─────────────┼────────────┼────────────────────┼─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

### Question 3: Top 5 Longest Juz by Verse Count

Find the 5 Juz with the most verses.

**SQL Concepts:** `ORDER BY` sorts the results; `LIMIT` keeps the top five.

In [ ]:
%%script duckdb
-- Read the Juz records and sort them by verse count.
SELECT
    juz_number,
    verses_count,
    first_verse_key,
    last_verse_key
FROM 'quran-data/juz.json'
-- DESC puts the largest counts first; LIMIT keeps only five rows.
ORDER BY verses_count DESC
LIMIT 5;

┌────────────┬──────────────┬─────────────┬──────────────────┬───────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────┬───────────┬────────────────┬─────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│ juz_number │ verses_count │ start_surah │ start_surah_name │ start_aya │                                             start_aya_text                                              │ end_surah │ end_surah_name │ end_aya │                                               end_aya_text                                                │
│   int64    │    int64     │    int32    │     varchar      │   int32   │                                                 varchar                                                 │   int32   │    varchar     │  int32  │                                                  varchar                                                  │
├────────────┼──

### Question 4: Top 5 Surahs by Total Word Count

Find the 5 largest Surahs by total word count.

**SQL Concepts:** `JOIN`, `GROUP BY`, `SUM`.

In [ ]:
%%script duckdb
-- Join surahs to ayahs and total the words in each surah.
SELECT
    su.name_english AS surah_name,
    su.revelation_place,
    SUM(a.words_count) AS total_words
FROM 'quran-data/surah.json' AS su
JOIN 'quran-data/ayah.json' AS a
    ON su.id = a.surah_number
GROUP BY su.name_english, su.revelation_place
ORDER BY total_words DESC
LIMIT 5;

### Question 5: Find Verse Matches (Mutashabihat)

Find verses with high match coverage (coverage >= 80).

**SQL Concepts:** `UNNEST`, `WHERE`, `ORDER BY`, `LIMIT`.

In [3]:
%%script duckdb
-- Name the unnested struct so its fields are easy to read.
SELECT
    a.ayah_key AS source_verse,
    matched.matched_ayah_key AS matched_verse,
    matched.coverage AS coverage
FROM 'quran-data/matching-ayah.json' AS a
CROSS JOIN UNNEST(a.matched_ayat) AS matches(matched)
WHERE matched.coverage >= 80
ORDER BY matched.coverage DESC
LIMIT 10;

┌──────────────┬───────────────┬──────────┐
│ source_verse │ matched_verse │ coverage │
│   varchar    │    varchar    │  int64   │
├──────────────┼───────────────┼──────────┤
│ 56:27        │ 56:38         │      200 │
│ 61:1         │ 57:1          │      122 │
│ 59:1         │ 57:1          │      122 │
│ 9:129        │ 27:26         │      113 │
│ 2:5          │ 31:5          │      100 │
│ 1:2          │ 37:182        │      100 │
│ 1:6          │ 4:68          │      100 │
│ 2:1          │ 3:1           │      100 │
│ 2:34         │ 20:116        │      100 │
│ 1:1          │ 1:3           │      100 │
└──────────────┴───────────────┴──────────┘
  10 rows                       3 columns


In [ ]:
%%script duckdb
-- Export the grouped result directly from the surah JSON file.
COPY (
    SELECT
        revelation_place,
        AVG(verses_count) AS avg_verse_count,
        COUNT(*) AS surah_count
    FROM 'quran-data/surah.json'
    GROUP BY revelation_place
    ORDER BY surah_count DESC
) TO 'quran-data/summary_by_revelation_place.parquet' (FORMAT parquet);